# HCP feature integration

Merge completed parcel-level RISE features into model-ready train and test Feather files.

In [ ]:
from pathlib import Path

# Resolve project data and configuration paths from the pipeline directory.
PROJECT_ROOT = Path('..').resolve()
import rise.integration as integration
from rise import fetch_feature_names
from rise.features import load_feature_config
from rise.integration import (
    find_missing_training_features,
    generate_group_mean_feather,
    generate_training_feather,
    read_subject_training_frame,
)
from rise.preprocessing import read_subject_ids
from rise.integration import read_training_subjects


### Integration settings

In [ ]:
# Feature-extraction configuration and integration paths.
CONFIG_PATH = PROJECT_ROOT / 'configs/hcp_feature_parameters.jsonc'
FEATURE_NAMES_FILE = fetch_feature_names()
TRAIN_SUBJECT_FILE = PROJECT_ROOT / 'data/HCP/info/train.txt'
TEST_SUBJECT_FILE = PROJECT_ROOT / 'data/HCP/info/test.txt'
INTEGRATION_OUTPUT_ROOT = PROJECT_ROOT / 'data/HCP/integrated_data_new'

# Root directory containing HCP individual feature files.
FEATURE_ROOT = PROJECT_ROOT / 'data/HCP/processed_data'

RUNS = [
    'mean',
    'ses-1_task-rest_run-1',
    'ses-1_task-rest_run-2',
    'ses-2_task-rest_run-1',
    'ses-2_task-rest_run-2',
]
MODES = ['NGR', 'GSR']


In [ ]:
# Load extraction settings, then add integration-specific settings.
config = load_feature_config(CONFIG_PATH)
PARCEL_METADATA_FILE = Path(config['atlas_mapping_file'])
config.update({
    'feature_names_file': str(FEATURE_NAMES_FILE),
    'parcel_metadata_file': str(PARCEL_METADATA_FILE),
    'train_subject_file': str(TRAIN_SUBJECT_FILE),
    'test_subject_file': str(TEST_SUBJECT_FILE),
    'training_output_root': str(INTEGRATION_OUTPUT_ROOT),
    'training_runs': RUNS,
    'training_modes': MODES,
})

train_subjects = read_training_subjects(TRAIN_SUBJECT_FILE)
test_subjects = read_training_subjects(TEST_SUBJECT_FILE)
print('Train subjects:', len(train_subjects))
print('Test subjects:', len(test_subjects))
print('Runs:', RUNS)
print('Output:', INTEGRATION_OUTPUT_ROOT)


## Validate one subject

In [ ]:
# Check required inputs and inspect one complete subject before integration.
preview_subject = train_subjects[0]
preview_run = RUNS[0]
missing = find_missing_training_features(
    [preview_subject], preview_run, config, feature_root=FEATURE_ROOT
)
if not missing.empty:
    raise FileNotFoundError(missing.to_string(index=False))

preview = read_subject_training_frame(
    preview_subject, preview_run, config, feature_root=FEATURE_ROOT
)
print('Preview subject:', preview_subject)
print('Preview shape:', preview.shape)
preview.iloc[:3, :8]


## Generate train and test Feather files

In [ ]:
# Keep the subject lists, runs, and output paths explicit here. For a
# multi-center dataset, pass a subject-to-feature-root mapping through
# feature_root=FEATURE_ROOT.
train_outputs = {}
for run in RUNS:
    train_outputs[run] = generate_training_feather(
        train_subjects,
        run,
        'train',
        config,
        feature_root=FEATURE_ROOT,
        show_progress=True,
    )

test_outputs = {}
for run in RUNS:
    test_outputs[run] = generate_training_feather(
        test_subjects,
        run,
        'test',
        config,
        feature_root=FEATURE_ROOT,
        show_progress=True,
    )

train_outputs


## Calculate group-mean features

Average the 378 features across subjects for each cortical parcel. Parcel metadata is retained and the group-level subject identifier is set to `000000`.

In [ ]:
# Define group-level aggregation only where it is used.
GROUPBY_COLUMN = 'Parcel label'
GROUP_SUBJECT_ID = '000000'


def generate_split_group_means(sample_outputs):
    """Generate group-mean features for one train/test split."""
    return {
        run: generate_group_mean_feather(
            sample_path,
            config,
            groupby_column=GROUPBY_COLUMN,
            group_subject_id=GROUP_SUBJECT_ID,
        )
        for run, sample_path in sample_outputs.items()
    }


train_group_outputs = generate_split_group_means(train_outputs)
test_group_outputs = generate_split_group_means(test_outputs)
train_group_outputs
